In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount= 'true')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd
import os

DATA_PATH = Path('/content/drive/MyDrive/O-List/')

files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

df = {}

for name, filename in files.items():
    df[name] = pd.read_csv(DATA_PATH / filename)

print('Dataset loaded Successfully!')

Dataset loaded Successfully!


In [ ]:
for name, data in df.items():
    print(f"{name:20} {data.shape}")

orders               (99441, 8)
order_items          (112650, 7)
order_payments       (103886, 5)
order_reviews        (99224, 7)
customers            (99441, 5)
products             (32951, 9)
sellers              (3095, 4)
geolocation          (1000163, 5)
category_translation (71, 2)


In [ ]:
inventory = pd.DataFrame([
    {
        "table": name,
        "rows": data.shape[0],
        "columns": data.shape[1],
        "memory_mb": round(data.memory_usage(deep=True).sum() / 1024**2, 2)
    }
    for name, data in df.items()
])

inventory.sort_values("rows", ascending=False)

,table,rows,columns,memory_mb
7,geolocation,1000163,5,129.38
1,order_items,112650,7,35.99
2,order_payments,103886,5,16.23
0,orders,99441,8,52.94
4,customers,99441,5,26.59
3,order_reviews,99224,7,39.12
5,products,32951,9,6.30
6,sellers,3095,4,0.59
8,category_translation,71,2,0.01


In [ ]:
schema = []

for name, data in df.items():
    for col in data.columns:
        schema.append({
            "table": name,
            "column": col,
            "dtype": str(data[col].dtype),
            "null_count": data[col].isna().sum(),
            "null_pct": round(data[col].isna().mean() * 100, 2),
            "unique_values": data[col].nunique()
        })

schema_df = pd.DataFrame(schema)

schema_df

,table,column,dtype,null_count,null_pct,unique_values
0,orders,order_id,object,0,0.00,99441
1,orders,customer_id,object,0,0.00,99441
2,orders,order_status,object,0,0.00,8
3,orders,order_purchase_timestamp,object,0,0.00,98875
4,orders,order_approved_at,object,160,0.16,90733
5,orders,order_delivered_carrier_date,object,1783,1.79,81018
6,orders,order_delivered_customer_date,object,2965,2.98,95664
7,orders,order_estimated_delivery_date,object,0,0.00,459
8,order_items,order_id,object,0,0.00,98666
9,order_items,order_item_id,int64,0,0.00,21


In [ ]:
# ==========================================
# PRIMARY KEY & DUPLICATE VALIDATION
# ==========================================

primary_keys = {
    "orders": ["order_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"]
}

print("PRIMARY KEY VALIDATION")
print("=" * 70)

for table, keys in primary_keys.items():
    data = df[table]

    duplicate_count = data.duplicated(subset=keys).sum()
    null_count = data[keys].isna().any(axis=1).sum()

    print(
        f"{table:25} | "
        f"Rows: {len(data):8,} | "
        f"Duplicate keys: {duplicate_count:6,} | "
        f"Null keys: {null_count:6,}"
    )


print("\nORDER ITEMS COMPOSITE KEY")
print("=" * 70)

order_items = df["order_items"]

duplicate_item_keys = order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

print(
    f"Duplicate (order_id, order_item_id): "
    f"{duplicate_item_keys:,}"
)


print("\nPAYMENT COMPOSITE KEY")
print("=" * 70)

payments = df["order_payments"]

duplicate_payment_keys = payments.duplicated(
    subset=["order_id", "payment_sequential"]
).sum()

print(
    f"Duplicate (order_id, payment_sequential): "
    f"{duplicate_payment_keys:,}"
)


print("\nREVIEW ID")
print("=" * 70)

reviews = df["order_reviews"]

print(
    f"Duplicate review_id: "
    f"{reviews['review_id'].duplicated().sum():,}"
)

PRIMARY KEY VALIDATION
orders                    | Rows:   99,441 | Duplicate keys:      0 | Null keys:      0
customers                 | Rows:   99,441 | Duplicate keys:      0 | Null keys:      0
products                  | Rows:   32,951 | Duplicate keys:      0 | Null keys:      0
sellers                   | Rows:    3,095 | Duplicate keys:      0 | Null keys:      0
category_translation      | Rows:       71 | Duplicate keys:      0 | Null keys:      0

ORDER ITEMS COMPOSITE KEY
Duplicate (order_id, order_item_id): 0

PAYMENT COMPOSITE KEY
Duplicate (order_id, payment_sequential): 0

REVIEW ID
Duplicate review_id: 814


In [ ]:
# ==========================================
# INVESTIGATE DUPLICATE REVIEW IDs
# ==========================================

reviews = df["order_reviews"]

duplicate_reviews = reviews[
    reviews["review_id"].duplicated(keep=False)
].sort_values("review_id")

print("Rows involved in duplicate review_ids:",
      len(duplicate_reviews))

print("Unique duplicated review_ids:",
      duplicate_reviews["review_id"].nunique())

duplicate_reviews.head(20)

Rows involved in duplicate review_ids: 1603
Unique duplicated review_ids: 789


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [ ]:
# How many duplicated review IDs are associated
# with multiple orders?

review_id_order_counts = (
    duplicate_reviews
    .groupby("review_id")["order_id"]
    .nunique()
)

print(
    "Duplicated review_ids associated with >1 order:",
    (review_id_order_counts > 1).sum()
)

print(
    "Duplicated review_ids associated with exactly 1 order:",
    (review_id_order_counts == 1).sum()
)

Duplicated review_ids associated with >1 order: 789
Duplicated review_ids associated with exactly 1 order: 0


In [ ]:
# Are the duplicated rows completely identical?

duplicate_review_rows = (
    duplicate_reviews
    .duplicated(keep=False)
)

print(
    "Completely duplicated rows:",
    duplicate_review_rows.sum()
)

Completely duplicated rows: 0


In [ ]:
# ==========================================
# REVIEW COMPOSITE KEY VALIDATION
# ==========================================

reviews = df["order_reviews"]

duplicate_review_order_pairs = reviews.duplicated(
    subset=["review_id", "order_id"]
).sum()

print(
    "Duplicate (review_id, order_id) pairs:",
    duplicate_review_order_pairs
)

Duplicate (review_id, order_id) pairs: 0


In [ ]:
review_counts = (
    reviews.groupby("order_id")
           .size()
           .sort_values(ascending=False)
)

print("Orders with more than one review record:",
      (review_counts > 1).sum())

print("\nMaximum reviews for a single order:",
      review_counts.max())

print("\nDistribution:")
print(review_counts.value_counts().sort_index())

Orders with more than one review record: 547

Maximum reviews for a single order: 3

Distribution:
1    98126
2      543
3        4
Name: count, dtype: int64


In [ ]:
# ==========================================
# FOREIGN-KEY INTEGRITY VALIDATION
# ==========================================

orders = df["orders"]
customers = df["customers"]
items = df["order_items"]
payments = df["order_payments"]
reviews = df["order_reviews"]
products = df["products"]
sellers = df["sellers"]

print("FOREIGN-KEY VALIDATION")
print("=" * 75)

# Orders → Customers
invalid_order_customers = ~orders["customer_id"].isin(
    customers["customer_id"]
)

print(
    f"Orders → Customers: "
    f"{invalid_order_customers.sum():,} invalid references"
)


# Order Items → Orders
invalid_item_orders = ~items["order_id"].isin(
    orders["order_id"]
)

print(
    f"Order Items → Orders: "
    f"{invalid_item_orders.sum():,} invalid references"
)


# Order Items → Products
invalid_item_products = ~items["product_id"].isin(
    products["product_id"]
)

print(
    f"Order Items → Products: "
    f"{invalid_item_products.sum():,} invalid references"
)


# Order Items → Sellers
invalid_item_sellers = ~items["seller_id"].isin(
    sellers["seller_id"]
)

print(
    f"Order Items → Sellers: "
    f"{invalid_item_sellers.sum():,} invalid references"
)


# Payments → Orders
invalid_payment_orders = ~payments["order_id"].isin(
    orders["order_id"]
)

print(
    f"Payments → Orders: "
    f"{invalid_payment_orders.sum():,} invalid references"
)


# Reviews → Orders
invalid_review_orders = ~reviews["order_id"].isin(
    orders["order_id"]
)

print(
    f"Reviews → Orders: "
    f"{invalid_review_orders.sum():,} invalid references"
)

FOREIGN-KEY VALIDATION
Orders → Customers: 0 invalid references
Order Items → Orders: 0 invalid references
Order Items → Products: 0 invalid references
Order Items → Sellers: 0 invalid references
Payments → Orders: 0 invalid references
Reviews → Orders: 0 invalid references


In [ ]:
# ==========================================
# RELATIONSHIP COVERAGE ANALYSIS
# ==========================================

print("RELATIONSHIP COVERAGE")
print("=" * 75)

# Unique IDs at child level
item_orders = set(items["order_id"].unique())
payment_orders = set(payments["order_id"].unique())
review_orders = set(reviews["order_id"].unique())

# Orders with / without child records
total_orders = orders["order_id"].nunique()

orders_without_items = total_orders - len(
    set(orders["order_id"]) & item_orders
)

orders_without_payments = total_orders - len(
    set(orders["order_id"]) & payment_orders
)

orders_without_reviews = total_orders - len(
    set(orders["order_id"]) & review_orders
)

print(f"Total orders:              {total_orders:,}")
print(f"Orders without items:      {orders_without_items:,}")
print(f"Orders without payments:   {orders_without_payments:,}")
print(f"Orders without reviews:    {orders_without_reviews:,}")

RELATIONSHIP COVERAGE
Total orders:              99,441
Orders without items:      775
Orders without payments:   1
Orders without reviews:    768


In [ ]:
# ==========================================
# PRODUCT & SELLER COVERAGE
# ==========================================

sold_products = items["product_id"].nunique()
total_products = products["product_id"].nunique()

active_sellers = items["seller_id"].nunique()
total_sellers = sellers["seller_id"].nunique()

print("PRODUCT COVERAGE")
print("=" * 75)

print(f"Total products:             {total_products:,}")
print(f"Products with sales:        {sold_products:,}")
print(f"Products never sold:        {total_products - sold_products:,}")

print("\nSELLER COVERAGE")
print("=" * 75)

print(f"Total sellers:              {total_sellers:,}")
print(f"Sellers with sales:         {active_sellers:,}")
print(f"Sellers with no sales:      {total_sellers - active_sellers:,}")

PRODUCT COVERAGE
Total products:             32,951
Products with sales:        32,951
Products never sold:        0

SELLER COVERAGE
Total sellers:              3,095
Sellers with sales:         3,095
Sellers with no sales:      0


In [ ]:
# ==========================================
# ORDER STATUS VS RECORD COVERAGE
# ==========================================

order_coverage = orders[[
    "order_id",
    "order_status"
]].copy()

order_coverage["has_items"] = (
    order_coverage["order_id"].isin(item_orders)
)

order_coverage["has_payment"] = (
    order_coverage["order_id"].isin(payment_orders)
)

order_coverage["has_review"] = (
    order_coverage["order_id"].isin(review_orders)
)

coverage_by_status = (
    order_coverage
    .groupby("order_status")
    .agg(
        orders=("order_id", "count"),
        orders_with_items=("has_items", "sum"),
        orders_with_payment=("has_payment", "sum"),
        orders_with_review=("has_review", "sum")
    )
)

coverage_by_status

,orders,orders_with_items,orders_with_payment,orders_with_review
order_status,,,,
approved,2,2,2,2
canceled,625,461,625,605
created,5,0,5,3
delivered,96478,96478,96477,95832
invoiced,314,312,314,309
processing,301,301,301,295
shipped,1107,1106,1107,1032
unavailable,609,6,609,595


In [ ]:
# ==========================================
# IDENTIFY ORDERS WITHOUT PAYMENTS
# ==========================================

orders_without_payment = orders[
    ~orders["order_id"].isin(payment_orders)
]

orders_without_payment[
    ["order_id", "customer_id", "order_status"]
]

,order_id,customer_id,order_status
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered


In [ ]:
# ==========================================
# TEMPORAL VALIDATION
# ==========================================

orders = df["orders"].copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert to datetime
for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

print("DATE RANGES")
print("=" * 75)

for col in date_columns:
    print(
        f"{col:35} "
        f"{orders[col].min()} → {orders[col].max()}"
    )

DATE RANGES
order_purchase_timestamp            2016-09-04 21:15:19 → 2018-10-17 17:30:18
order_approved_at                   2016-09-15 12:16:38 → 2018-09-03 17:40:06
order_delivered_carrier_date        2016-10-08 10:34:01 → 2018-09-11 19:48:28
order_delivered_customer_date       2016-10-11 13:46:32 → 2018-10-17 13:22:46
order_estimated_delivery_date       2016-09-30 00:00:00 → 2018-11-12 00:00:00


In [ ]:
# ==========================================
# ORDER LIFECYCLE LOGIC CHECKS
# ==========================================

temporal_checks = pd.DataFrame({

    "approval_before_purchase":
        orders["order_approved_at"]
        < orders["order_purchase_timestamp"],

    "carrier_before_purchase":
        orders["order_delivered_carrier_date"]
        < orders["order_purchase_timestamp"],

    "delivery_before_purchase":
        orders["order_delivered_customer_date"]
        < orders["order_purchase_timestamp"],

    "delivery_before_carrier":
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
})

print("TEMPORAL LOGIC VIOLATIONS")
print("=" * 75)

for col in temporal_checks.columns:
    print(
        f"{col:35} "
        f"{temporal_checks[col].sum():,}"
    )

TEMPORAL LOGIC VIOLATIONS
approval_before_purchase            0
carrier_before_purchase             166
delivery_before_purchase            0
delivery_before_carrier             23


In [ ]:
# ==========================================
# INVESTIGATE CARRIER-BEFORE-PURCHASE CASES
# ==========================================

carrier_before_purchase = orders[
    orders["order_delivered_carrier_date"]
    < orders["order_purchase_timestamp"]
].copy()

print("Number of violations:",
      len(carrier_before_purchase))

carrier_before_purchase[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].head(20)

Number of violations: 166


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
1111,ad133696906f6a78826daa0911b7daec,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37,2018-07-18
1329,74e033208dc13a7b8127eb8e73d09b76,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36,2018-05-29
1372,a6b58794fd2ba533359a76c08df576e3,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32,2018-06-08
1864,5792e0b1c8c8a2bf53af468c9a422c88,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02,2018-08-09
2760,c3eb293fd154223498b6551a728203e8,delivered,2018-07-19 14:06:04,2018-07-19 14:22:51,2018-07-19 13:49:00,2018-07-24 19:35:36,2018-08-06
3473,b0c2a7d04b165525254254a728c50a4e,delivered,2018-06-07 13:28:30,2018-06-07 13:57:22,2018-06-07 13:22:00,2018-06-21 17:36:43,2018-07-04
3661,2033a4586b5bec3229ebc1675a8ae092,delivered,2018-06-12 10:10:25,2018-06-12 10:39:59,2018-06-12 10:09:00,2018-06-19 14:08:27,2018-07-19
4114,08adcddad19d3acf37d1fa01cb9ded1e,delivered,2018-06-27 11:16:44,2018-06-27 11:30:56,2018-06-27 10:57:00,2018-06-29 17:39:53,2018-07-18
4159,dee6298ce7d1fb2645141ef9972157aa,shipped,2018-04-30 14:06:12,2018-04-30 14:15:24,2018-04-30 12:59:00,NaT,2018-05-28


In [ ]:
carrier_before_purchase["violation_hours"] = (
    carrier_before_purchase["order_purchase_timestamp"]
    - carrier_before_purchase["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

carrier_before_purchase["violation_hours"].describe()

,violation_hours
count,166.000000
mean,26.039478
std,318.923845
min,0.006667
25%,0.234444
50%,0.599861
75%,1.017500
max,4109.098056


In [ ]:
# ==========================================
# INVESTIGATE DELIVERY-BEFORE-CARRIER CASES
# ==========================================

delivery_before_carrier = orders[
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
].copy()

print("Number of violations:",
      len(delivery_before_carrier))

delivery_before_carrier[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].head(20)

Number of violations: 23


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18


In [ ]:
delivery_before_carrier["violation_hours"] = (
    delivery_before_carrier["order_delivered_carrier_date"]
    - delivery_before_carrier["order_delivered_customer_date"]
).dt.total_seconds() / 3600

delivery_before_carrier["violation_hours"].describe()

,violation_hours
count,23.000000
mean,78.455133
std,89.308070
min,0.388333
25%,23.338333
50%,39.864444
75%,128.759028
max,386.308056


In [ ]:
# ==========================================
# EXTREME TEMPORAL VIOLATIONS
# ==========================================

carrier_before_purchase[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "violation_hours"
    ]
].sort_values(
    "violation_hours",
    ascending=False
).head(10)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,violation_hours
25883,7c48bb55e8e4f7e56d412e9653db37bc,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,2018-08-07,4109.098056
83321,4021cd7611d6d9ce5ffcd24817fc374f,delivered,2018-08-18 11:49:40,2018-08-18 12:10:30,2018-08-14 06:22:00,2018-08-29 04:41:53,2018-08-30,101.461111
67844,db090a16182b263b1e896bb26c6f66cf,delivered,2018-07-13 16:14:08,2018-07-13 16:44:09,2018-07-13 13:59:00,2018-07-27 11:34:31,2018-08-06,2.252222
65452,9711d975b961355b4b5d636857e48498,delivered,2018-06-13 15:23:50,2018-06-13 15:38:18,2018-06-13 13:15:00,2018-06-26 00:33:48,2018-06-29,2.147222
79401,89d32b64af005178b318f76cd60f2c3c,delivered,2018-07-06 11:54:40,2018-07-06 12:11:05,2018-07-06 09:48:00,2018-07-10 11:44:45,2018-07-27,2.111111
13969,6192897f85cb2aff6dd1fca56fddb45e,delivered,2018-07-18 13:34:22,2018-07-18 13:50:12,2018-07-18 11:33:00,2018-07-26 15:02:53,2018-08-03,2.022778
43657,3ed559dec69ab96d9074fedaf18650eb,delivered,2018-06-15 13:20:09,2018-06-15 13:44:37,2018-06-15 11:22:00,2018-06-26 15:53:25,2018-07-23,1.969167
4256,4e157a36ea9cf89bde6fff57a780b525,delivered,2018-08-24 14:37:50,2018-08-24 14:50:14,2018-08-24 12:43:00,2018-08-30 20:47:26,2018-09-10,1.913889
74503,119060af27b04f5fae8b2b0e27eef7f9,delivered,2018-06-11 14:34:47,2018-06-12 09:31:54,2018-06-11 12:45:00,2018-06-14 22:18:58,2018-07-05,1.829722
21910,e23c9dcc304042fa90d538be679fa68d,delivered,2018-06-13 11:41:23,2018-06-13 13:31:40,2018-06-13 09:53:00,2018-06-25 13:57:31,2018-07-11,1.806389


In [ ]:
delivery_before_carrier[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "violation_hours"
    ]
].sort_values(
    "violation_hours",
    ascending=False
)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,violation_hours
34939,c1e2bf2b7dd3309f2f5356c6b63968fa,delivered,2017-02-10 10:19:10,2017-02-10 10:30:13,2017-03-02 17:34:26,2017-02-14 15:15:57,2017-03-15,386.308056
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18,189.095000
45302,29941903985f944b0ffc49c479c1547d,delivered,2017-05-29 16:16:50,2017-05-29 16:25:16,2017-06-09 15:07:29,2017-06-02 11:09:16,2017-06-23,171.970278
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11,144.238889
49933,76458889992169d3135b264dc13aec67,delivered,2016-10-07 10:05:16,2016-10-07 11:24:43,2016-10-26 11:43:06,2016-10-20 18:03:17,2016-11-29,137.663611
71227,19feb5627c41ea1b36a8e50a469b3644,delivered,2016-10-07 17:09:56,2016-10-07 17:32:09,2016-10-26 11:42:05,2016-10-20 19:07:54,2016-12-01,136.569722
74967,d5558a097766363b8e76b38c43332e8a,delivered,2017-02-04 19:01:33,2017-02-04 19:31:12,2017-02-15 08:55:26,2017-02-10 07:58:32,2017-03-01,120.948333
41636,b866af202be0692766081310cd4085e1,delivered,2017-01-27 14:59:17,2017-01-27 15:30:46,2017-02-20 02:32:08,2017-02-15 03:53:46,2017-04-17,118.639444
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14,69.417222
78556,ea1dcb4757a844d2642547797bd5feb0,delivered,2017-07-18 13:38:29,2017-07-18 13:50:23,2017-07-27 19:21:31,2017-07-25 19:43:10,2017-08-15,47.639167


In [ ]:
# ==========================================
# TEMPORAL QUALITY FLAGS
# ==========================================

orders["flag_carrier_before_purchase"] = (
    orders["order_delivered_carrier_date"]
    < orders["order_purchase_timestamp"]
)

orders["flag_delivery_before_carrier"] = (
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
)

orders["flag_temporal_violation"] = (
    orders["flag_carrier_before_purchase"]
    | orders["flag_delivery_before_carrier"]
)

print(
    "Orders with temporal violations:",
    orders["flag_temporal_violation"].sum()
)

Orders with temporal violations: 189


In [ ]:
# ==========================================
# DELIVERY DURATION METRICS
# ==========================================

orders["purchase_to_carrier_days"] = (
    orders["order_delivered_carrier_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

orders["carrier_to_delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_delivered_carrier_date"]
).dt.total_seconds() / (60 * 60 * 24)

orders["purchase_to_delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

In [ ]:
orders["valid_purchase_to_carrier"] = (
    orders["order_purchase_timestamp"].notna()
    & orders["order_delivered_carrier_date"].notna()
    & (orders["purchase_to_carrier_days"] >= 0)
)

orders["valid_carrier_to_delivery"] = (
    orders["order_delivered_carrier_date"].notna()
    & orders["order_delivered_customer_date"].notna()
    & (orders["carrier_to_delivery_days"] >= 0)
)

orders["valid_purchase_to_delivery"] = (
    orders["order_purchase_timestamp"].notna()
    & orders["order_delivered_customer_date"].notna()
    & (orders["purchase_to_delivery_days"] >= 0)
)

In [ ]:
valid_delivery = orders[
    (orders["order_status"] == "delivered")
    & orders["valid_purchase_to_delivery"]
].copy()

valid_delivery["purchase_to_delivery_days"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

,purchase_to_delivery_days
count,96470.000000
mean,12.558217
std,9.546156
min,0.533414
1%,1.825151
5%,3.015196
25%,6.766204
50%,10.217477
75%,15.720182
95%,29.274046


In [ ]:
valid_delivery["purchase_to_delivery_days"].sort_values(
    ascending=False
).head(20)

,purchase_to_delivery_days
19590,209.628611
55619,208.351759
61610,195.634016
70307,194.850174
89130,194.633611
38509,194.049583
11399,191.463542
81401,189.863160
54480,188.134618
68769,187.743843


In [ ]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

In [ ]:
valid_estimated_delivery = orders[
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
].copy()

valid_estimated_delivery["delivery_delay_days"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

,delivery_delay_days
count,96470.000000
mean,-11.178126
std,10.184354
min,-146.016123
1%,-35.144500
5%,-25.939634
25%,-16.244065
50%,-11.948102
75%,-6.389815
95%,3.816993


In [ ]:
# ==========================================
# RECREATE VALID DELIVERY DATASET
# ==========================================

valid_delivery = orders[
    (orders["order_status"] == "delivered")
    & orders["valid_purchase_to_delivery"]
].copy()

valid_delivery["delivery_delay_days"] = (
    valid_delivery["order_delivered_customer_date"]
    - valid_delivery["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

extreme_delivery = valid_delivery[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "purchase_to_delivery_days",
        "delivery_delay_days"
    ]
].sort_values(
    "purchase_to_delivery_days",
    ascending=False
)

extreme_delivery.head(20)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_to_delivery_days,delivery_delay_days
19590,ca07593549f1816d26a572e06dc1eab6,delivered,2017-02-21 23:31:27,2017-02-23 02:35:15,2017-03-08 13:47:46,2017-09-19 14:36:39,2017-03-22,209.628611,181.608785
55619,1b3190b2dfa9d789e1f14c05b647a14a,delivered,2018-02-23 14:57:35,2018-02-23 15:16:14,2018-02-26 18:49:07,2018-09-19 23:24:07,2018-03-15,208.351759,188.975081
61610,440d0d17af552815d15a9e41abe49359,delivered,2017-03-07 23:59:51,2017-03-09 01:11:33,2017-03-15 13:00:08,2017-09-19 15:12:50,2017-04-07,195.634016,165.633912
70307,2fb597c2f772eca01b1f5c561bf6cc7b,delivered,2017-03-08 18:09:02,2017-03-08 18:09:02,2017-03-13 08:54:02,2017-09-19 14:33:17,2017-04-17,194.850174,155.606447
89130,285ab9426d6982034523a855f55a885e,delivered,2017-03-08 22:47:40,2017-03-08 22:47:40,2017-03-09 08:53:20,2017-09-19 14:00:04,2017-04-06,194.633611,166.583380
38509,0f4519c5f1c541ddec9f21b3bddd533a,delivered,2017-03-09 13:26:57,2017-03-09 13:26:57,2017-03-22 05:28:14,2017-09-19 14:38:21,2017-04-11,194.049583,161.609965
11399,47b40429ed8cce3aee9199792275433f,delivered,2018-01-03 09:44:01,2018-01-03 10:31:15,2018-02-06 01:48:28,2018-07-13 20:51:31,2018-01-19,191.463542,175.869109
81401,2fe324febf907e3ea3f2aa9650869fa5,delivered,2017-03-13 20:17:10,2017-03-13 20:17:10,2017-03-17 07:23:53,2017-09-19 17:00:07,2017-04-05,189.863160,167.708414
54480,2d7561026d542c8dbd8f0daeadf67a43,delivered,2017-03-15 11:24:27,2017-03-15 11:24:27,2017-03-16 09:29:16,2017-09-19 14:38:18,2017-04-13,188.134618,159.609931
68769,c27815f7e3dd0b926b58552628481575,delivered,2017-03-15 23:23:17,2017-03-15 23:23:17,2017-03-21 14:50:25,2017-09-19 17:14:25,2017-04-10,187.743843,162.718345


In [ ]:
# ==========================================
# DELIVERY PERFORMANCE CLASSIFICATION
# ==========================================
import numpy as np

valid_estimated_delivery["actual_delivery_date"] = (
    valid_estimated_delivery[
        "order_delivered_customer_date"
    ].dt.date
)

valid_estimated_delivery["estimated_delivery_date"] = (
    valid_estimated_delivery[
        "order_estimated_delivery_date"
    ].dt.date
)

valid_estimated_delivery["delivery_performance"] = np.select(
    [
        valid_estimated_delivery["actual_delivery_date"]
        < valid_estimated_delivery["estimated_delivery_date"],

        valid_estimated_delivery["actual_delivery_date"]
        == valid_estimated_delivery["estimated_delivery_date"],

        valid_estimated_delivery["actual_delivery_date"]
        > valid_estimated_delivery["estimated_delivery_date"]
    ],
    [
        "Early",
        "On Time",
        "Late"
    ],
    default="Unknown"
)

valid_estimated_delivery[
    "delivery_performance"
].value_counts()

,count
delivery_performance,
Early,88644
Late,6534
On Time,1292


In [ ]:
delivery_performance_summary = (
    valid_estimated_delivery[
        "delivery_performance"
    ]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

delivery_performance_summary

,proportion
delivery_performance,
Early,91.89
Late,6.77
On Time,1.34


In [ ]:
# ==========================================
# EXTREME DELIVERY DATE CONCENTRATION
# ==========================================

extreme_delivery["delivery_date"] = (
    extreme_delivery["order_delivered_customer_date"].dt.date
)

print("Most common delivery dates among extreme deliveries:")
print(
    extreme_delivery["delivery_date"]
    .value_counts()
    .head(15)
)



Most common delivery dates among extreme deliveries:
delivery_date
2018-08-27    446
2018-08-13    442
2018-05-14    434
2018-05-21    431
2018-05-18    425
2018-04-11    413
2017-12-11    412
2018-07-03    410
2018-05-03    409
2017-06-19    405
2018-07-30    405
2017-12-19    401
2018-08-23    396
2018-08-28    395
2018-08-20    393
Name: count, dtype: int64


In [ ]:
delivery_date_counts = (
    valid_delivery["order_delivered_customer_date"]
    .dt.date
    .value_counts()
)

delivery_date_counts.head(20)

,count
order_delivered_customer_date,
2018-08-27,446
2018-08-13,442
2018-05-14,434
2018-05-21,431
2018-05-18,425
2018-04-11,413
2017-12-11,412
2018-07-03,410
2018-05-03,409


In [ ]:
# ==========================================
# EXTREME DELIVERY TAIL
# ==========================================

p99 = valid_delivery["purchase_to_delivery_days"].quantile(0.99)

extreme_delivery = valid_delivery[
    valid_delivery["purchase_to_delivery_days"] > p99
].copy()

print(f"99th percentile: {p99:.2f} days")
print(f"Orders above P99: {len(extreme_delivery):,}")

extreme_delivery[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "purchase_to_delivery_days",
        "delivery_delay_days"
    ]
].sort_values(
    "purchase_to_delivery_days",
    ascending=False
).head(20)

99th percentile: 46.05 days
Orders above P99: 965


,order_id,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_to_delivery_days,delivery_delay_days
19590,ca07593549f1816d26a572e06dc1eab6,2017-02-21 23:31:27,2017-03-08 13:47:46,2017-09-19 14:36:39,2017-03-22,209.628611,181.608785
55619,1b3190b2dfa9d789e1f14c05b647a14a,2018-02-23 14:57:35,2018-02-26 18:49:07,2018-09-19 23:24:07,2018-03-15,208.351759,188.975081
61610,440d0d17af552815d15a9e41abe49359,2017-03-07 23:59:51,2017-03-15 13:00:08,2017-09-19 15:12:50,2017-04-07,195.634016,165.633912
70307,2fb597c2f772eca01b1f5c561bf6cc7b,2017-03-08 18:09:02,2017-03-13 08:54:02,2017-09-19 14:33:17,2017-04-17,194.850174,155.606447
89130,285ab9426d6982034523a855f55a885e,2017-03-08 22:47:40,2017-03-09 08:53:20,2017-09-19 14:00:04,2017-04-06,194.633611,166.583380
38509,0f4519c5f1c541ddec9f21b3bddd533a,2017-03-09 13:26:57,2017-03-22 05:28:14,2017-09-19 14:38:21,2017-04-11,194.049583,161.609965
11399,47b40429ed8cce3aee9199792275433f,2018-01-03 09:44:01,2018-02-06 01:48:28,2018-07-13 20:51:31,2018-01-19,191.463542,175.869109
81401,2fe324febf907e3ea3f2aa9650869fa5,2017-03-13 20:17:10,2017-03-17 07:23:53,2017-09-19 17:00:07,2017-04-05,189.863160,167.708414
54480,2d7561026d542c8dbd8f0daeadf67a43,2017-03-15 11:24:27,2017-03-16 09:29:16,2017-09-19 14:38:18,2017-04-13,188.134618,159.609931
68769,c27815f7e3dd0b926b58552628481575,2017-03-15 23:23:17,2017-03-21 14:50:25,2017-09-19 17:14:25,2017-04-10,187.743843,162.718345


In [ ]:
# Delivery dates among actual extreme orders

extreme_delivery[
    "order_delivered_customer_date"
].dt.date.value_counts().head(20)

,count
order_delivered_customer_date,
2017-09-19,33
2018-04-19,17
2018-04-18,16
2018-05-03,14
2018-01-16,14
2018-04-17,14
2018-04-20,14
2018-04-16,13
2018-04-24,12


In [ ]:
# Compare top delivery dates

comparison = pd.DataFrame({
    "all_deliveries": valid_delivery[
        "order_delivered_customer_date"
    ].dt.date.value_counts(),

    "extreme_deliveries": extreme_delivery[
        "order_delivered_customer_date"
    ].dt.date.value_counts()
}).fillna(0)

comparison["extreme_share"] = (
    comparison["extreme_deliveries"]
    / comparison["all_deliveries"]
    * 100
)

comparison.sort_values(
    "extreme_deliveries",
    ascending=False
).head(20)

,all_deliveries,extreme_deliveries,extreme_share
order_delivered_customer_date,,,
2017-09-19,282,33.0,11.702128
2018-04-19,265,17.0,6.415094
2018-04-18,353,16.0,4.532578
2018-04-17,321,14.0,4.361371
2018-05-03,409,14.0,3.422983
2018-01-16,280,14.0,5.000000
2018-04-20,326,14.0,4.294479
2018-04-16,335,13.0,3.880597
2018-01-23,328,12.0,3.658537


In [ ]:
# ==========================================
# ITEM-LEVEL FINANCIAL VALIDATION
# ==========================================

items = df["order_items"].copy()

print("ITEM-LEVEL FINANCIAL VALIDATION")
print("=" * 70)

print(f"Negative prices:         {(items['price'] < 0).sum():,}")
print(f"Zero prices:             {(items['price'] == 0).sum():,}")
print(f"Negative freight:        {(items['freight_value'] < 0).sum():,}")
print(f"Zero freight:            {(items['freight_value'] == 0).sum():,}")

print("\nPRICE DISTRIBUTION")
print("=" * 70)
print(items["price"].describe())

print("\nFREIGHT DISTRIBUTION")
print("=" * 70)
print(items["freight_value"].describe())

ITEM-LEVEL FINANCIAL VALIDATION
Negative prices:         0
Zero prices:             0
Negative freight:        0
Zero freight:            383

PRICE DISTRIBUTION
count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

FREIGHT DISTRIBUTION
count    112650.000000
mean         19.990320
std          15.806405
min           0.000000
25%          13.080000
50%          16.260000
75%          21.150000
max         409.680000
Name: freight_value, dtype: float64


In [ ]:
# ==========================================
# PAYMENT VALIDATION
# ==========================================

payments = df["order_payments"].copy()

print("PAYMENT VALIDATION")
print("=" * 70)

print(f"Negative payment values: {(payments['payment_value'] < 0).sum():,}")
print(f"Zero payment values:     {(payments['payment_value'] == 0).sum():,}")

print("\nPAYMENT VALUE DISTRIBUTION")
print("=" * 70)
print(payments["payment_value"].describe())

print("\nPAYMENT TYPES")
print("=" * 70)
print(payments["payment_type"].value_counts(dropna=False))

print("\nPAYMENT INSTALLMENTS")
print("=" * 70)
print(payments["payment_installments"].describe())

print("\nInstallments < 1:")
print(
    (payments["payment_installments"] < 1).sum()
)

PAYMENT VALIDATION
Negative payment values: 0
Zero payment values:     9

PAYMENT VALUE DISTRIBUTION
count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

PAYMENT TYPES
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

PAYMENT INSTALLMENTS
count    103886.000000
mean          2.853349
std           2.687051
min           0.000000
25%           1.000000
50%           1.000000
75%           4.000000
max          24.000000
Name: payment_installments, dtype: float64

Installments < 1:
2


In [ ]:
payment_installment_summary = (
    payments
    .groupby("payment_type")
    .agg(
        transactions=("payment_sequential", "count"),
        min_installments=("payment_installments", "min"),
        max_installments=("payment_installments", "max"),
        avg_installments=("payment_installments", "mean"),
        total_value=("payment_value", "sum")
    )
    .sort_values("transactions", ascending=False)
)

payment_installment_summary

,transactions,min_installments,max_installments,avg_installments,total_value
payment_type,,,,,
credit_card,76795,0,24,3.507155,12542084.19
boleto,19784,1,1,1.000000,2869361.27
voucher,5775,1,1,1.000000,379436.87
debit_card,1529,1,1,1.000000,217989.79
not_defined,3,1,1,1.000000,0.00


In [ ]:
# ==========================================
# ORDER-LEVEL ITEM TOTALS
# ==========================================

item_totals = (
    items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        item_revenue=("price", "sum"),
        freight_revenue=("freight_value", "sum")
    )
    .reset_index()
)

item_totals["order_item_total"] = (
    item_totals["item_revenue"]
    + item_totals["freight_revenue"]
)

item_totals.head()

,order_id,item_count,item_revenue,freight_revenue,order_item_total
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04


In [ ]:
# ==========================================
# ORDER-LEVEL PAYMENT TOTALS
# ==========================================

payment_totals = (
    payments
    .groupby("order_id")
    .agg(
        payment_count=("payment_sequential", "count"),
        payment_total=("payment_value", "sum")
    )
    .reset_index()
)

payment_totals.head()

,order_id,payment_count,payment_total
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,259.83
2,000229ec398224ef6ca0657da4fc703e,1,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04


In [ ]:
# ==========================================
# ITEM VS PAYMENT RECONCILIATION
# ==========================================

financial_reconciliation = (
    item_totals
    .merge(
        payment_totals,
        on="order_id",
        how="outer",
        indicator=True
    )
)

financial_reconciliation["difference"] = (
    financial_reconciliation["payment_total"]
    - financial_reconciliation["order_item_total"]
)

financial_reconciliation.head()


,order_id,item_count,item_revenue,freight_revenue,order_item_total,payment_count,payment_total,_merge,difference
0,00010242fe8c5a6d1ba2dd792cb16214,1.0,58.90,13.29,72.19,1.0,72.19,both,0.000000e+00
1,00018f77f2f0320c557190d7a144bdd3,1.0,239.90,19.93,259.83,1.0,259.83,both,0.000000e+00
2,000229ec398224ef6ca0657da4fc703e,1.0,199.00,17.87,216.87,1.0,216.87,both,0.000000e+00
3,00024acbcdf0a6daa1e931b038114c75,1.0,12.99,12.79,25.78,1.0,25.78,both,0.000000e+00
4,00042b26cf59d7ce69dfabb4e55b4fd9,1.0,199.90,18.14,218.04,1.0,218.04,both,-2.842171e-14


In [ ]:

financial_reconciliation["difference"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

,difference
count,9.866500e+04
mean,2.909228e-02
std,1.129221e+00
min,-5.162000e+01
1%,-5.684342e-14
5%,-1.421085e-14
25%,0.000000e+00
50%,0.000000e+00
75%,0.000000e+00
95%,1.421085e-14


In [ ]:
financial_reconciliation["abs_difference"] = (
    financial_reconciliation["difference"].abs()
)

print(
    "Exact matches:",
    (financial_reconciliation["abs_difference"] < 0.01).sum()
)

print(
    "Differences > ₹0.01:",
    (financial_reconciliation["abs_difference"] >= 0.01).sum()
)

print(
    "Differences > ₹1:",
    (financial_reconciliation["abs_difference"] > 1).sum()
)

print(
    "Differences > ₹10:",
    (financial_reconciliation["abs_difference"] > 10).sum()
)

Exact matches: 98284
Differences > ₹0.01: 381
Differences > ₹1: 249
Differences > ₹10: 98


In [ ]:
# ==========================================
# FINANCIAL RECONCILIATION ANOMALIES
# ==========================================

reconciliation_issues = (
    financial_reconciliation[
        financial_reconciliation["abs_difference"] > 1
    ]
    .sort_values(
        "abs_difference",
        ascending=False
    )
)

print(
    f"Orders with payment/item difference > ₹1: "
    f"{len(reconciliation_issues):,}"
)

reconciliation_issues[
    [
        "order_id",
        "item_count",
        "item_revenue",
        "freight_revenue",
        "order_item_total",
        "payment_count",
        "payment_total",
        "difference"
    ]
].head(20)

Orders with payment/item difference > ₹1: 249


,order_id,item_count,item_revenue,freight_revenue,order_item_total,payment_count,payment_total,difference
80176,ce6d150fb29ada17d2082f4847107665,1.0,1299.00,104.66,1403.66,1.0,1586.47,182.81
42865,6e5fe7366a2e1bfbf3257dba0af1267f,6.0,179.19,108.72,287.91,1.0,406.92,119.01
43791,70b742795bc441e94a44a084b6d9ce7a,1.0,269.99,196.94,466.93,1.0,578.82,111.89
59228,996c7e73600ad3723e8627ab7bef81e4,1.0,559.90,28.00,587.90,1.0,664.43,76.53
43794,70b7e94ea46d3e8b5bc12a50186edaf0,3.0,167.88,45.27,213.15,1.0,274.84,61.69
73090,bc2c82b0ef78d2252b6176d1972db7c9,3.0,165.00,77.01,242.01,1.0,303.02,61.01
68015,af9ffff2ce6b3defd34fd4c78857a379,1.0,395.65,17.52,413.17,1.0,466.97,53.80
14745,262118ce178bb3e4590a3adcf6d62e6b,2.0,119.80,57.94,177.74,1.0,126.12,-51.62
74503,bfdb5bbb06458d600a33d61f5f287472,1.0,297.00,51.93,348.93,1.0,394.36,45.43
54744,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,2.0,209.80,44.65,254.45,1.0,293.89,39.44


In [ ]:
# ==========================================
# PAYMENT TYPE FOR RECONCILIATION ISSUES
# ==========================================

payment_type_by_order = (
    payments
    .groupby("order_id")["payment_type"]
    .agg(lambda x: ", ".join(sorted(x.dropna().unique())))
    .reset_index()
)

reconciliation_with_type = (
    reconciliation_issues
    .merge(
        payment_type_by_order,
        on="order_id",
        how="left"
    )
)

reconciliation_with_type[
    "payment_type"
].value_counts(dropna=False)

,count
payment_type,
credit_card,237
debit_card,7
"credit_card, voucher",5


In [ ]:
# ==========================================
# ZERO / INVALID PAYMENT RECORDS
# ==========================================

print("ZERO PAYMENT VALUES")
print("=" * 70)

zero_payments = payments[
    payments["payment_value"] == 0
]

zero_payments

ZERO PAYMENT VALUES


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [ ]:
print("ZERO INSTALLMENT RECORDS")
print("=" * 70)

zero_installments = payments[
    payments["payment_installments"] < 1
]

zero_installments

ZERO INSTALLMENT RECORDS


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [ ]:
print("UNDEFINED PAYMENT TYPE")
print("=" * 70)

payments[
    payments["payment_type"] == "not_defined"
]

UNDEFINED PAYMENT TYPE


,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [ ]:
# ==========================================
# INVESTIGATE RECONCILIATION ANOMALIES
# ==========================================

issue_order_ids = reconciliation_issues["order_id"]

issue_payments = (
    payments[
        payments["order_id"].isin(issue_order_ids)
    ]
    .sort_values(["order_id", "payment_sequential"])
)

issue_payments.head(50)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
97517,00789ce015e7e5791c7914f32bb4fad4,1,credit_card,10,190.81
2943,016726239765c18f66826453f39c64e3,1,credit_card,10,265.77
15829,01e51b7c3025655646143d09b911e1d7,1,credit_card,3,35.02
6548,028aa7c930356788f861ed1b7f984819,1,credit_card,6,62.94
39782,02f4dd90ba0feb8ec394cac05862d2b5,1,credit_card,6,141.65
11055,033ccfbdfc4d29677b7e1e6df3a82820,1,credit_card,3,30.82
37684,033ccfbdfc4d29677b7e1e6df3a82820,2,credit_card,3,38.44
83151,0345ba423eed2b009e0e407b19c422e4,1,credit_card,5,83.17
28383,03b218d39c422c250f389120c531b61f,1,credit_card,5,58.03
53921,04e00ba23c33890eaee39b02e8185cc2,1,credit_card,5,53.41


In [ ]:
issue_payment_counts = (
    issue_payments
    .groupby("order_id")
    .size()
)

print("Payment-count distribution for reconciliation anomalies:")
print(issue_payment_counts.value_counts().sort_index())

Payment-count distribution for reconciliation anomalies:
1    238
2     11
Name: count, dtype: int64


In [ ]:
# Compare payment count between anomalous and normal orders

print(
    "Average payment records - anomalous orders:",
    issue_payment_counts.mean()
)

print(
    "Orders with multiple payment records:",
    (issue_payment_counts > 1).sum()
)

Average payment records - anomalous orders: 1.0441767068273093
Orders with multiple payment records: 11


In [ ]:
# ==========================================
# RECONCILIATION ISSUES BY ORDER STATUS
# ==========================================

reconciliation_status = (
    reconciliation_issues
    .merge(
        orders[["order_id", "order_status"]],
        on="order_id",
        how="left"
    )
)

print(
    reconciliation_status["order_status"]
    .value_counts()
)

print("\nPercentage:")
print(
    reconciliation_status["order_status"]
    .value_counts(normalize=True).mul(100).round(2)
)

order_status
delivered    246
canceled       2
shipped        1
Name: count, dtype: int64

Percentage:
order_status
delivered    98.8
canceled      0.8
shipped       0.4
Name: proportion, dtype: float64


In [ ]:
# ==========================================
# DIRECTION OF FINANCIAL DIFFERENCES
# ==========================================

reconciliation_status["difference_direction"] = np.select(
    [
        reconciliation_status["difference"] > 1,
        reconciliation_status["difference"] < -1
    ],
    [
        "Payment > Item Total",
        "Payment < Item Total"
    ],
    default="Approximately Equal"
)

print(
    reconciliation_status["difference_direction"]
    .value_counts()
)

difference_direction
Payment > Item Total    232
Payment < Item Total     17
Name: count, dtype: int64


In [ ]:
# ==========================================
# LARGEST RECONCILIATION DIFFERENCES
# ==========================================

reconciliation_status[
    [
        "order_id",
        "order_status",
        "item_count",
        "order_item_total",
        "payment_count",
        "payment_total",
        "difference"
    ]
].sort_values(
    "difference",
    key=lambda x: x.abs(),
    ascending=False
).head(20)

,order_id,order_status,item_count,order_item_total,payment_count,payment_total,difference
0,ce6d150fb29ada17d2082f4847107665,delivered,1.0,1403.66,1.0,1586.47,182.81
1,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,6.0,287.91,1.0,406.92,119.01
2,70b742795bc441e94a44a084b6d9ce7a,delivered,1.0,466.93,1.0,578.82,111.89
3,996c7e73600ad3723e8627ab7bef81e4,delivered,1.0,587.90,1.0,664.43,76.53
4,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,3.0,213.15,1.0,274.84,61.69
5,bc2c82b0ef78d2252b6176d1972db7c9,delivered,3.0,242.01,1.0,303.02,61.01
6,af9ffff2ce6b3defd34fd4c78857a379,delivered,1.0,413.17,1.0,466.97,53.80
7,262118ce178bb3e4590a3adcf6d62e6b,delivered,2.0,177.74,1.0,126.12,-51.62
8,bfdb5bbb06458d600a33d61f5f287472,delivered,1.0,348.93,1.0,394.36,45.43
9,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,delivered,2.0,254.45,1.0,293.89,39.44


In [ ]:
# ==========================================
# RELATIVE RECONCILIATION DIFFERENCE
# ==========================================

reconciliation_status["difference_pct"] = (
    reconciliation_status["difference"]
    / reconciliation_status["order_item_total"]
    * 100
)

reconciliation_status[
    [
        "order_id",
        "order_status",
        "order_item_total",
        "payment_total",
        "difference",
        "difference_pct"
    ]
].sort_values(
    "difference_pct",
    key=lambda x: x.abs(),
    ascending=False
).head(20)

,order_id,order_status,order_item_total,payment_total,difference,difference_pct
1,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,287.91,406.92,119.01,41.335834
7,262118ce178bb3e4590a3adcf6d62e6b,delivered,177.74,126.12,-51.62,-29.042422
4,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,213.15,274.84,61.69,28.942060
43,8c82eab3813ca92415655a5e4b1a8b7a,delivered,67.73,87.32,19.59,28.923668
16,e8295bee3fcc58cbd94f165f36c11f4b,delivered,117.85,151.93,34.08,28.918116
98,aa6bd33ba1853d846d3085a88ae37083,delivered,35.14,25.14,-10.00,-28.457598
5,bc2c82b0ef78d2252b6176d1972db7c9,delivered,242.01,303.02,61.01,25.209702
49,6b956d19010e956e03b175e66b2b7142,delivered,75.48,93.58,18.10,23.979862
58,971d58b94f7bda9a79f4847e1509a433,delivered,67.67,83.89,16.22,23.969263
2,70b742795bc441e94a44a084b6d9ce7a,delivered,466.93,578.82,111.89,23.962907


In [ ]:
print(
    reconciliation_status["difference_pct"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
)

count    249.000000
mean       9.469374
std        7.519744
min      -29.042422
1%       -11.939802
5%        -4.393758
25%        5.800604
50%       10.418536
75%       13.031824
95%       15.512708
99%       28.921003
max       41.335834
Name: difference_pct, dtype: float64


In [ ]:
# ==========================================
# REVIEW DATA VALIDATION
# ==========================================

reviews = df["order_reviews"].copy()

print("REVIEW DATA VALIDATION")
print("=" * 70)

print(f"Invalid review scores (<1 or >5): "
      f"{((reviews['review_score'] < 1) | (reviews['review_score'] > 5)).sum():,}")

print(f"Missing review scores: "
      f"{reviews['review_score'].isna().sum():,}")

print("\nREVIEW SCORE DISTRIBUTION")
print("=" * 70)

print(
    reviews["review_score"]
    .value_counts()
    .sort_index()
)

REVIEW DATA VALIDATION
Invalid review scores (<1 or >5): 0
Missing review scores: 0

REVIEW SCORE DISTRIBUTION
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [ ]:
# ==========================================
# REVIEW TEXT COMPLETENESS
# ==========================================

review_text_status = pd.DataFrame({
    "missing_title": reviews["review_comment_title"].isna(),
    "missing_message": reviews["review_comment_message"].isna()
})

print("Missing review title:",
      review_text_status["missing_title"].sum())

print("Missing review message:",
      review_text_status["missing_message"].sum())

print(
    "Both title and message missing:",
    (
        review_text_status["missing_title"]
        & review_text_status["missing_message"]
    ).sum()
)

print(
    "At least one text field present:",
    (
        ~(
            review_text_status["missing_title"]
            & review_text_status["missing_message"]
        )
    ).sum()
)

Missing review title: 87656
Missing review message: 58247
Both title and message missing: 56518
At least one text field present: 42706


In [ ]:
reviews["has_written_feedback"] = (
    reviews["review_comment_title"].notna()
    | reviews["review_comment_message"].notna()
)

reviews.groupby("has_written_feedback")["review_score"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
has_written_feedback,,,
False,56518,4.375916,5.0
True,42706,3.703297,5.0


In [ ]:
products = df["products"].copy()

print("PRODUCT DATA VALIDATION")
print("=" * 70)

print("Total products:", len(products))

print("\nMissing values:")
print(
    products.isna()
    .sum()
    .sort_values(ascending=False)
)

PRODUCT DATA VALIDATION
Total products: 32951

Missing values:
product_category_name         610
product_description_lenght    610
product_name_lenght           610
product_photos_qty            610
product_weight_g                2
product_height_cm               2
product_length_cm               2
product_width_cm                2
product_id                      0
dtype: int64


In [ ]:
product_attributes = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

missing_counts = products[product_attributes].isna().sum(axis=1)

print("Missing attribute count per product:")
print(missing_counts.value_counts().sort_index())

print("\nProducts missing all 4 attributes:",
      (missing_counts == 4).sum())

print("Products missing only some attributes:",
      ((missing_counts > 0) & (missing_counts < 4)).sum())

Missing attribute count per product:
0    32341
4      610
Name: count, dtype: int64

Products missing all 4 attributes: 610
Products missing only some attributes: 0


In [ ]:
physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print("PHYSICAL ATTRIBUTE VALIDATION")
print("=" * 70)

for col in physical_columns:
    print(
        f"{col}: "
        f"negative = {(products[col] < 0).sum():,}, "
        f"zero = {(products[col] == 0).sum():,}, "
        f"missing = {products[col].isna().sum():,}"
    )

PHYSICAL ATTRIBUTE VALIDATION
product_weight_g: negative = 0, zero = 4, missing = 2
product_length_cm: negative = 0, zero = 0, missing = 2
product_height_cm: negative = 0, zero = 0, missing = 2
product_width_cm: negative = 0, zero = 0, missing = 2


In [ ]:
zero_weight_products = products[
    products["product_weight_g"] == 0
]

zero_weight_products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


In [ ]:
physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products[physical_columns].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32949.000000,32949.000000,32949.000000,32949.000000
mean,2276.472488,30.815078,16.937661,23.196728
std,4282.038731,16.914458,13.637554,12.079047
min,0.000000,7.000000,2.000000,6.000000
1%,60.000000,16.000000,2.000000,11.000000
5%,105.000000,16.000000,3.000000,11.000000
25%,300.000000,18.000000,8.000000,15.000000
50%,700.000000,25.000000,13.000000,20.000000
75%,1900.000000,38.000000,21.000000,30.000000
95%,10850.000000,65.000000,44.000000,47.000000


In [ ]:
translation = df["category_translation"].copy()

print("CATEGORY TRANSLATION VALIDATION")
print("=" * 70)

print("Translation rows:", len(translation))

print(
    "Duplicate Portuguese categories:",
    translation["product_category_name"].duplicated().sum()
)

print(
    "Duplicate English categories:",
    translation["product_category_name_english"].duplicated().sum()
)

CATEGORY TRANSLATION VALIDATION
Translation rows: 71
Duplicate Portuguese categories: 0
Duplicate English categories: 0


In [ ]:
product_categories = set(
    products["product_category_name"]
    .dropna()
    .unique()
)

translated_categories = set(
    translation["product_category_name"]
    .dropna()
    .unique()
)

untranslated_categories = product_categories - translated_categories

print("Unique product categories:", len(product_categories))
print("Translated categories:",
      len(product_categories & translated_categories))
print("Untranslated categories:",
      len(untranslated_categories))

print("\nUntranslated categories:")
print(sorted(untranslated_categories))

Unique product categories: 73
Translated categories: 71
Untranslated categories: 2

Untranslated categories:
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


In [ ]:
# ==========================================
# CLEANED DATASETS
# ==========================================

orders_clean = df["orders"].copy()
order_items_clean = df["order_items"].copy()
payments_clean = df["order_payments"].copy()
reviews_clean = df["order_reviews"].copy()
customers_clean = df["customers"].copy()
products_clean = df["products"].copy()
sellers_clean = df["sellers"].copy()
translation_clean = df["category_translation"].copy()

In [ ]:
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_cols:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce"
    )

In [ ]:
review_date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_date_cols:
    reviews_clean[col] = pd.to_datetime(
        reviews_clean[col],
        errors="coerce"
    )

In [ ]:
products_clean = products_clean.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

In [ ]:
products_clean.loc[
    products_clean["product_weight_g"] <= 0,
    "product_weight_g"
] = np.nan

In [ ]:
category_map = dict(
    zip(
        translation_clean["product_category_name"],
        translation_clean["product_category_name_english"]
    )
)

In [ ]:
products_clean["product_category_english"] = (
    products_clean["product_category_name"]
    .map(category_map)
)

In [ ]:
manual_category_map = {
    "pc_gamer": "Gaming PC",
    "portateis_cozinha_e_preparadores_de_alimentos":
        "Kitchen & Food Preparation Appliances"
}

products_clean["product_category_english"] = (
    products_clean["product_category_english"]
    .fillna(
        products_clean["product_category_name"]
        .map(manual_category_map)
    )
)

In [ ]:
products_clean["product_category_english"] = (
    products_clean["product_category_english"]
    .fillna("Unknown")
)

In [ ]:
orders_clean["approval_valid"] = (
    orders_clean["order_approved_at"]
    >= orders_clean["order_purchase_timestamp"]
)

orders_clean["carrier_delivery_valid"] = (
    orders_clean["order_delivered_carrier_date"]
    >= orders_clean["order_purchase_timestamp"]
)

orders_clean["customer_delivery_valid"] = (
    orders_clean["order_delivered_customer_date"]
    >= orders_clean["order_purchase_timestamp"]
)

orders_clean["delivery_sequence_valid"] = (
    orders_clean["order_delivered_customer_date"]
    >= orders_clean["order_delivered_carrier_date"]
)

In [ ]:
orders_clean["temporal_valid"] = (
    orders_clean["approval_valid"]
    & orders_clean["carrier_delivery_valid"]
    & orders_clean["customer_delivery_valid"]
    & orders_clean["delivery_sequence_valid"]
)

In [ ]:
valid_delivery = orders_clean[
    orders_clean["order_status"].eq("delivered")
    & orders_clean["temporal_valid"]
].copy()

In [ ]:
valid_delivery["purchase_to_delivery_days"] = (
    valid_delivery["order_delivered_customer_date"]
    - valid_delivery["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [ ]:
valid_delivery["delivery_delay_days"] = (
    valid_delivery["order_delivered_customer_date"]
    - valid_delivery["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

In [ ]:
valid_delivery["delivery_performance"] = np.select(
    [
        valid_delivery["delivery_delay_days"] < 0,
        valid_delivery["delivery_delay_days"] == 0,
        valid_delivery["delivery_delay_days"] > 0
    ],
    [
        "Early",
        "On Time",
        "Late"
    ],
    default="Unknown"
)

In [ ]:
# Aggregate item-level revenue to order level

order_item_totals = (
    order_items_clean
    .groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        item_revenue=("price", "sum"),
        freight_revenue=("freight_value", "sum")
    )
)

order_item_totals["order_item_total"] = (
    order_item_totals["item_revenue"]
    + order_item_totals["freight_revenue"]
)

In [ ]:
order_payment_totals = (
    payments_clean
    .groupby("order_id", as_index=False)
    .agg(
        payment_count=("payment_sequential", "count"),
        payment_total=("payment_value", "sum")
    )
)

In [ ]:
financial_check = order_item_totals.merge(
    order_payment_totals,
    on="order_id",
    how="outer",
    indicator=True
)

financial_check["difference"] = (
    financial_check["payment_total"]
    - financial_check["order_item_total"]
)

In [ ]:
financial_check["financial_anomaly"] = (
    financial_check["difference"].abs() > 1
)

In [ ]:
financial_check["difference_direction"] = np.select(
    [
        financial_check["difference"] > 1,
        financial_check["difference"] < -1
    ],
    [
        "Payment > Item Total",
        "Payment < Item Total"
    ],
    default="Match"
)

In [ ]:
print(
    financial_check["difference_direction"]
    .value_counts()
)

print(
    "\nFinancial anomalies:",
    financial_check["financial_anomaly"].sum()
)

difference_direction
Match                   99192
Payment > Item Total      232
Payment < Item Total       17
Name: count, dtype: int64

Financial anomalies: 249


In [ ]:
import os

output_path = "/content/drive/MyDrive/O-List_cleaned"

os.makedirs(output_path, exist_ok=True)

orders_clean.to_csv(
    f"{output_path}/orders_clean.csv",
    index=False
)

order_items_clean.to_csv(
    f"{output_path}/order_items_clean.csv",
    index=False
)

payments_clean.to_csv(
    f"{output_path}/order_payments_clean.csv",
    index=False
)

reviews_clean.to_csv(
    f"{output_path}/order_reviews_clean.csv",
    index=False
)

customers_clean.to_csv(
    f"{output_path}/customers_clean.csv",
    index=False
)

products_clean.to_csv(
    f"{output_path}/products_clean.csv",
    index=False
)

sellers_clean.to_csv(
    f"{output_path}/sellers_clean.csv",
    index=False
)

translation_clean.to_csv(
    f"{output_path}/category_translation_clean.csv",
    index=False
)

print("Cleaned datasets successfully saved to:")
print(output_path)

Cleaned datasets successfully saved to:
/content/drive/MyDrive/O-List_cleaned


In [ ]:
# ============================================================
# ADD FINAL ANALYTICAL FEATURES
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. ORDER DELIVERY FEATURES
# ------------------------------------------------------------

orders_clean["purchase_to_delivery_days"] = (
    orders_clean["order_delivered_customer_date"]
    - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

orders_clean["delivery_delay_days"] = (
    orders_clean["order_delivered_customer_date"]
    - orders_clean["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)


# ------------------------------------------------------------
# 2. DELIVERY PERFORMANCE
# ------------------------------------------------------------

orders_clean["actual_delivery_date"] = (
    orders_clean["order_delivered_customer_date"].dt.date
)

orders_clean["estimated_delivery_date"] = (
    orders_clean["order_estimated_delivery_date"].dt.date
)

orders_clean["delivery_performance"] = np.select(
    [
        orders_clean["actual_delivery_date"]
        < orders_clean["estimated_delivery_date"],

        orders_clean["actual_delivery_date"]
        == orders_clean["estimated_delivery_date"],

        orders_clean["actual_delivery_date"]
        > orders_clean["estimated_delivery_date"]
    ],
    [
        "Early",
        "On Time",
        "Late"
    ],
    default="Unknown"
)


# ------------------------------------------------------------
# 3. REVIEW FEEDBACK FLAG
# ------------------------------------------------------------

reviews_clean["has_written_feedback"] = (
    reviews_clean["review_comment_title"].notna()
    | reviews_clean["review_comment_message"].notna()
)


# ------------------------------------------------------------
# 4. CHECK
# ------------------------------------------------------------

print("ORDER FEATURES")
print("-" * 40)

for col in [
    "purchase_to_delivery_days",
    "delivery_delay_days",
    "delivery_performance"
]:
    print(f"✓ {col}: {col in orders_clean.columns}")


print("\nREVIEW FEATURES")
print("-" * 40)

print(
    f"✓ has_written_feedback: "
    f"{'has_written_feedback' in reviews_clean.columns}"
)

print("\nDelivery Performance Distribution:")
print(
    orders_clean["delivery_performance"]
    .value_counts(dropna=False)
)

ORDER FEATURES
----------------------------------------
✓ purchase_to_delivery_days: True
✓ delivery_delay_days: True
✓ delivery_performance: True

REVIEW FEATURES
----------------------------------------
✓ has_written_feedback: True

Delivery Performance Distribution:
delivery_performance
Early      88649
Late        6535
Unknown     2965
On Time     1292
Name: count, dtype: int64


In [ ]:
orders_clean.drop(
    columns=[
        "actual_delivery_date",
        "estimated_delivery_date"
    ],
    inplace=True
)

In [ ]:
# ============================================================
# FINAL CLEAN DATA VALIDATION & PREVIEW
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# 1. DATASET REGISTRY
# ------------------------------------------------------------

datasets = {
    "orders": orders_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "products": products_clean,
    "sellers": sellers_clean,
    "customers": customers_clean,
    "category_translation": translation_clean
}

# ------------------------------------------------------------
# 2. BASIC VALIDATION
# ------------------------------------------------------------

print("=" * 75)
print("FINAL CLEAN DATA VALIDATION")
print("=" * 75)

for name, df in datasets.items():

    print(f"\n{name.upper()}")
    print("-" * 50)

    print(f"Rows       : {len(df):,}")
    print(f"Columns    : {len(df.columns)}")
    print(f"Duplicates : {df.duplicated().sum():,}")
    print(f"Missing    : {df.isna().sum().sum():,}")

# ------------------------------------------------------------
# 3. PRIMARY KEY CHECK
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PRIMARY KEY CHECK")
print("=" * 75)

primary_keys = {
    "orders": "order_id",
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "products": "product_id",
    "sellers": "seller_id"
}

for table, key in primary_keys.items():

    df = datasets[table]

    if isinstance(key, list):
        duplicates = df.duplicated(subset=key).sum()
        nulls = df[key].isna().any(axis=1).sum()
    else:
        duplicates = df[key].duplicated().sum()
        nulls = df[key].isna().sum()

    print(
        f"{table:<20} "
        f"duplicate keys: {duplicates:<6} "
        f"null keys: {nulls}"
    )

# ------------------------------------------------------------
# 4. DERIVED FEATURES CHECK
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("DERIVED FEATURES")
print("=" * 75)

features = {
    "orders": [
        "purchase_to_delivery_days",
        "delivery_delay_days",
        "delivery_performance"
    ],
    "reviews": [
        "has_written_feedback"
    ],
    "products": [
        "product_category_english"
    ]
}

for table, columns in features.items():

    df = datasets[table]

    print(f"\n{table.upper()}")

    for col in columns:
        if col in df.columns:
            print(f"✓ {col}")
        else:
            print(f"✗ {col} — NOT FOUND")

# ------------------------------------------------------------
# 5. CLEANED DATA PREVIEW
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CLEANED DATA PREVIEW")
print("=" * 75)

for name, df in datasets.items():

    print(f"\n{'=' * 75}")
    print(f"{name.upper()} — {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("=" * 75)

    # Show first 5 rows
    display(df.head())

# ------------------------------------------------------------
# 6. COLUMN LIST
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL COLUMN STRUCTURE")
print("=" * 75)

for name, df in datasets.items():

    print(f"\n{name.upper()}")
    print("-" * 50)

    for col in df.columns:
        print(f"• {col}")

# ------------------------------------------------------------
# 7. EXPORT CLEAN DATA TO GOOGLE DRIVE
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("EXPORTING CLEAN DATA")
print("=" * 75)

output_path = "/content/drive/MyDrive/O-List_cleaned"

os.makedirs(output_path, exist_ok=True)

for name, df in datasets.items():

    path = f"{output_path}/{name}_clean.csv"

    df.to_csv(path, index=False)

    print(f"✓ {path}")

print("\nClean data export completed.")

FINAL CLEAN DATA VALIDATION

ORDERS
--------------------------------------------------
Rows       : 99,441
Columns    : 16
Duplicates : 0
Missing    : 10,838

ORDER_ITEMS
--------------------------------------------------
Rows       : 112,650
Columns    : 7
Duplicates : 0
Missing    : 0

PAYMENTS
--------------------------------------------------
Rows       : 103,886
Columns    : 5
Duplicates : 0
Missing    : 0

REVIEWS
--------------------------------------------------
Rows       : 99,224
Columns    : 8
Duplicates : 0
Missing    : 145,903

PRODUCTS
--------------------------------------------------
Rows       : 32,951
Columns    : 10
Duplicates : 0
Missing    : 2,452

SELLERS
--------------------------------------------------
Rows       : 3,095
Columns    : 4
Duplicates : 0
Missing    : 0

CUSTOMERS
--------------------------------------------------
Rows       : 99,441
Columns    : 5
Duplicates : 0
Missing    : 0

CATEGORY_TRANSLATION
--------------------------------------------------

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_valid,carrier_delivery_valid,customer_delivery_valid,delivery_sequence_valid,temporal_valid,purchase_to_delivery_days,delivery_delay_days,delivery_performance
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,True,True,True,True,8.436574,-7.107488,Early
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,True,True,True,True,True,13.782037,-5.355729,Early
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,True,True,True,True,9.394213,-17.245498,Early
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,True,True,True,True,True,13.208750,-12.980069,Early
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,True,True,True,True,True,2.873877,-9.238171,Early



ORDER_ITEMS — 112,650 rows × 7 columns


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14



PAYMENTS — 103,886 rows × 5 columns


...


CUSTOMERS — 99,441 rows × 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



CATEGORY_TRANSLATION — 71 rows × 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor



FINAL COLUMN STRUCTURE

ORDERS
--------------------------------------------------
• order_id
• customer_id
• order_status
• order_purchase_timestamp
• order_approved_at
• order_delivered_carrier_date
• order_delivered_customer_date
• order_estimated_delivery_date
• approval_valid
• carrier_delivery_valid
• customer_delivery_valid
• delivery_sequence_valid
• temporal_valid
• purchase_to_delivery_days
• delivery_delay_days
• delivery_performance

ORDER_ITEMS
--------------------------------------------------
• order_id
• order_item_id
• product_id
• seller_id
• shipping_limit_date
• price
• freight_value

PAYMENTS
--------------------------------------------------
• order_id
• payment_sequential
• payment_type
• payment_installments
• payment_value

REVIEWS
--------------------------------------------------
• review_id
• order_id
• review_score
• review_comment_title
• review_comment_message
• review_creation_date
• review_answer_timestamp
• has_written_feedback

PRODUCTS
--------------

In [ ]:
# Re-export updated cleaned datasets

output_path = "/content/drive/MyDrive/O-List_cleaned"

for name, df in datasets.items():
    path = f"{output_path}/{name}_clean.csv"
    df.to_csv(path, index=False)
    print(f"✓ {path}")

print("\nUpdated clean data exported successfully.")

✓ /content/drive/MyDrive/O-List_cleaned/orders_clean.csv
✓ /content/drive/MyDrive/O-List_cleaned/order_items_clean.csv
✓ /content/drive/MyDrive/O-List_cleaned/payments_clean.csv
✓ /content/drive/MyDrive/O-List_cleaned/reviews_clean.csv
✓ /content/drive/MyDrive/O-List_cleaned/products_clean.csv
✓ /content/drive/MyDrive/O-List_cleaned/sellers_clean.csv
✓ /content/drive/MyDrive/O-List_cleaned/customers_clean.csv
✓ /content/drive/MyDrive/O-List_cleaned/category_translation_clean.csv

Updated clean data exported successfully.


In [ ]:
# ============================================================
# VERIFY SAVED ORDERS_CLEAN
# ============================================================

import pandas as pd

orders_check = pd.read_csv(
    "/content/drive/MyDrive/O-List_cleaned/orders_clean.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

print("=" * 70)
print("SAVED ORDERS_CLEAN VERIFICATION")
print("=" * 70)

print(f"\nShape: {orders_check.shape}")

print("\nRequired derived columns:")
print("-" * 50)

required_columns = [
    "approval_valid",
    "carrier_delivery_valid",
    "customer_delivery_valid",
    "delivery_sequence_valid",
    "temporal_valid",
    "purchase_to_delivery_days",
    "delivery_delay_days",
    "delivery_performance"
]

for col in required_columns:
    print(f"{'✓' if col in orders_check.columns else '✗'} {col}")

print("\nDelivery Performance:")
print("-" * 50)
print(
    orders_check["delivery_performance"]
    .value_counts(dropna=False)
)

print("\nPreview:")
display(
    orders_check[
        [
            "order_id",
            "order_status",
            "purchase_to_delivery_days",
            "delivery_delay_days",
            "delivery_performance"
        ]
    ].head(10)
)

SAVED ORDERS_CLEAN VERIFICATION

Shape: (99441, 16)

Required derived columns:
--------------------------------------------------
✓ approval_valid
✓ carrier_delivery_valid
✓ customer_delivery_valid
✓ delivery_sequence_valid
✓ temporal_valid
✓ purchase_to_delivery_days
✓ delivery_delay_days
✓ delivery_performance

Delivery Performance:
--------------------------------------------------
delivery_performance
Early      88649
Late        6535
Unknown     2965
On Time     1292
Name: count, dtype: int64

Preview:


,order_id,order_status,purchase_to_delivery_days,delivery_delay_days,delivery_performance
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,8.436574,-7.107488,Early
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,13.782037,-5.355729,Early
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,9.394213,-17.245498,Early
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,13.208750,-12.980069,Early
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2.873877,-9.238171,Early
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,16.542245,-5.543113,Early
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,NaN,NaN,Unknown
7,6514b8ad8028c9f2cc2374ded245783f,delivered,9.989826,-11.461215,Early
8,76c6e866289321a7c93b82b54852dc33,delivered,9.818762,-31.410995,Early
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,18.221852,-6.281597,Early


In [ ]:
# ============================================================
# VERIFY SAVED REVIEWS_CLEAN
# ============================================================

import pandas as pd

reviews_check = pd.read_csv(
    "/content/drive/MyDrive/O-List_cleaned/reviews_clean.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

print("=" * 70)
print("SAVED REVIEWS_CLEAN VERIFICATION")
print("=" * 70)

print(f"\nShape: {reviews_check.shape}")

print("\nRequired derived columns:")
print("-" * 50)

required_columns = [
    "has_written_feedback"
]

for col in required_columns:
    print(f"{'✓' if col in reviews_check.columns else '✗'} {col}")

# ------------------------------------------------------------
# REVIEW SCORE VALIDATION
# ------------------------------------------------------------

print("\nReview Score Distribution:")
print("-" * 50)

print(
    reviews_check["review_score"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# WRITTEN FEEDBACK DISTRIBUTION
# ------------------------------------------------------------

print("\nWritten Feedback Distribution:")
print("-" * 50)

print(
    reviews_check["has_written_feedback"]
    .value_counts(dropna=False)
)

# ------------------------------------------------------------
# PREVIEW
# ------------------------------------------------------------

print("\nPreview:")
display(
    reviews_check[
        [
            "review_id",
            "order_id",
            "review_score",
            "has_written_feedback",
            "review_comment_title",
            "review_comment_message"
        ]
    ].head(10)
)

SAVED REVIEWS_CLEAN VERIFICATION

Shape: (99224, 8)

Required derived columns:
--------------------------------------------------
✓ has_written_feedback

Review Score Distribution:
--------------------------------------------------
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Written Feedback Distribution:
--------------------------------------------------
has_written_feedback
False    56518
True     42706
Name: count, dtype: int64

Preview:


,review_id,order_id,review_score,has_written_feedback,review_comment_title,review_comment_message
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,False,NaN,NaN
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,False,NaN,NaN
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,False,NaN,NaN
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,True,NaN,Recebi bem antes do prazo estipulado.
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,True,NaN,Parabéns lojas lannister adorei comprar pela I...
5,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,False,NaN,NaN
6,07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,False,NaN,NaN
7,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,False,NaN,NaN
8,a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,False,NaN,NaN
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,True,recomendo,aparelho eficiente. no site a marca do aparelh...


In [ ]:
import os

output_path = "/content/drive/MyDrive/O-List_cleaned"

files = sorted(os.listdir(output_path))

print("Files in O-List_cleaned:")
print("=" * 60)

for i, file in enumerate(files, 1):
    print(f"{i}. {file}")

print(f"\nTotal files: {len(files)}")

Files in O-List_cleaned:
1. category_translation_clean.csv
2. customers_clean.csv
3. order_items_clean.csv
4. order_payments_clean.csv
5. order_reviews_clean.csv
6. orders_clean.csv
7. payments_clean.csv
8. products_clean.csv
9. reviews_clean.csv
10. sellers_clean.csv

Total files: 10


In [ ]:
# ============================================================
# FINAL CLEAN DATASET VALIDATION
# O-LIST E-COMMERCE PROJECT
# ============================================================

import os
import pandas as pd
import numpy as np

# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_PATH = "/content/drive/MyDrive/O-List_cleaned"

EXPECTED_FILES = {
    "orders": "orders_clean.csv",
    "order_items": "order_items_clean.csv",
    "payments": "payments_clean.csv",
    "reviews": "reviews_clean.csv",
    "products": "products_clean.csv",
    "sellers": "sellers_clean.csv",
    "customers": "customers_clean.csv",
    "category_translation": "category_translation_clean.csv"
}

# ============================================================
# 2. EXPECTED SCHEMA
# ============================================================

EXPECTED_COLUMNS = {

    "orders": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "approval_valid",
        "carrier_delivery_valid",
        "customer_delivery_valid",
        "delivery_sequence_valid",
        "temporal_valid",
        "purchase_to_delivery_days",
        "delivery_delay_days",
        "delivery_performance"
    ],

    "order_items": [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    ],

    "payments": [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ],

    "reviews": [
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp",
        "has_written_feedback"
    ],

    "products": [
        "product_id",
        "product_category_name",
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_category_english"
    ],

    "sellers": [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ],

    "customers": [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ],

    "category_translation": [
        "product_category_name",
        "product_category_name_english"
    ]
}

# ============================================================
# 3. LOAD DATA
# ============================================================

print("=" * 80)
print("FINAL CLEAN DATASET VALIDATION")
print("=" * 80)

print("\nLoading cleaned datasets...\n")

datasets = {}

for name, filename in EXPECTED_FILES.items():

    path = os.path.join(BASE_PATH, filename)

    if not os.path.exists(path):
        print(f"❌ MISSING FILE: {filename}")
    else:
        datasets[name] = pd.read_csv(path)
        print(f"✓ {filename:<40} {datasets[name].shape}")

# ============================================================
# 4. FILE COUNT CHECK
# ============================================================

print("\n" + "=" * 80)
print("1. FILE CHECK")
print("=" * 80)

actual_files = {
    f for f in os.listdir(BASE_PATH)
    if f.lower().endswith(".csv")
}

expected_files = set(EXPECTED_FILES.values())

unexpected_files = actual_files - expected_files
missing_files = expected_files - actual_files

if not missing_files and not unexpected_files:
    print("✓ Exactly the expected 8 CSV files are present.")
else:

    if missing_files:
        print("❌ Missing files:")
        for f in sorted(missing_files):
            print("   -", f)

    if unexpected_files:
        print("⚠ Unexpected files:")
        for f in sorted(unexpected_files):
            print("   -", f)

# ============================================================
# 5. SCHEMA CHECK
# ============================================================

print("\n" + "=" * 80)
print("2. SCHEMA CHECK")
print("=" * 80)

schema_pass = True

for name, expected_cols in EXPECTED_COLUMNS.items():

    if name not in datasets:
        continue

    actual_cols = list(datasets[name].columns)

    missing_cols = [
        col for col in expected_cols
        if col not in actual_cols
    ]

    extra_cols = [
        col for col in actual_cols
        if col not in expected_cols
    ]

    print(f"\n{name.upper()}")

    if not missing_cols:
        print("✓ Required columns present")
    else:
        schema_pass = False
        print("❌ Missing columns:")
        for col in missing_cols:
            print("   -", col)

    if extra_cols:
        print("ℹ Additional columns:")
        for col in extra_cols:
            print("   +", col)

# ============================================================
# 6. BASIC DATASET CHECK
# ============================================================

print("\n" + "=" * 80)
print("3. BASIC DATASET CHECK")
print("=" * 80)

for name, df in datasets.items():

    print(f"\n{name.upper()}")
    print("-" * 50)

    print(f"Rows        : {len(df):,}")
    print(f"Columns     : {len(df.columns)}")
    print(f"Duplicates  : {df.duplicated().sum():,}")
    print(f"Total nulls : {df.isna().sum().sum():,}")

# ============================================================
# 7. KEY VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("4. KEY VALIDATION")
print("=" * 80)

keys = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "customers": ["customer_id"],
    "category_translation": ["product_category_name"]
}

for table, key in keys.items():

    df = datasets[table]

    duplicate_keys = df.duplicated(
        subset=key
    ).sum()

    null_keys = df[key].isna().any(axis=1).sum()

    key_name = " + ".join(key)

    print(
        f"{table:<25} "
        f"Key: {key_name:<35} "
        f"Duplicates: {duplicate_keys:<6} "
        f"Nulls: {null_keys}"
    )

# ============================================================
# 8. ORDER BUSINESS RULES
# ============================================================

print("\n" + "=" * 80)
print("5. ORDER BUSINESS-RULE CHECKS")
print("=" * 80)

orders = datasets["orders"]

checks = {

    "Negative delivery duration":
        (orders["purchase_to_delivery_days"] < 0).sum(),

    "Negative delivery delay":
        (
            orders["delivery_delay_days"] < 0
        ).sum(),

    "Invalid delivery performance":
        (
            ~orders["delivery_performance"].isin(
                ["Early", "On Time", "Late", "Unknown"]
            )
        ).sum(),

    "Missing order_id":
        orders["order_id"].isna().sum(),

    "Missing customer_id":
        orders["customer_id"].isna().sum()
}

for check, value in checks.items():

    status = "✓" if value == 0 else "⚠"

    print(f"{status} {check:<40}: {value:,}")

# ============================================================
# 9. DELIVERY PERFORMANCE CONSISTENCY
# ============================================================

print("\n" + "=" * 80)
print("6. DELIVERY PERFORMANCE CHECK")
print("=" * 80)

valid_delivery = orders[
    orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
].copy()

valid_delivery["actual_date"] = pd.to_datetime(
    valid_delivery["order_delivered_customer_date"],
    errors="coerce"
).dt.date

valid_delivery["estimated_date"] = pd.to_datetime(
    valid_delivery["order_estimated_delivery_date"],
    errors="coerce"
).dt.date

expected_performance = np.select(
    [
        valid_delivery["actual_date"]
        < valid_delivery["estimated_date"],

        valid_delivery["actual_date"]
        == valid_delivery["estimated_date"],

        valid_delivery["actual_date"]
        > valid_delivery["estimated_date"]
    ],
    [
        "Early",
        "On Time",
        "Late"
    ],
    default="Unknown"
)

mismatch = (
    valid_delivery["delivery_performance"].values
    != expected_performance
).sum()

print(f"Performance mismatches: {mismatch:,}")

if mismatch == 0:
    print("✓ Delivery performance classifications are consistent.")
else:
    print("❌ Delivery performance has inconsistencies.")

# ============================================================
# 10. ORDER ITEMS CHECK
# ============================================================

print("\n" + "=" * 80)
print("7. ORDER ITEMS CHECK")
print("=" * 80)

items = datasets["order_items"]

print(
    f"Negative prices       : "
    f"{(items['price'] < 0).sum():,}"
)

print(
    f"Zero prices           : "
    f"{(items['price'] == 0).sum():,}"
)

print(
    f"Negative freight      : "
    f"{(items['freight_value'] < 0).sum():,}"
)

print(
    f"Zero freight          : "
    f"{(items['freight_value'] == 0).sum():,}"
)

# ============================================================
# 11. PAYMENT CHECK
# ============================================================

print("\n" + "=" * 80)
print("8. PAYMENT CHECK")
print("=" * 80)

payments = datasets["payments"]

print(
    f"Negative payment value : "
    f"{(payments['payment_value'] < 0).sum():,}"
)

print(
    f"Zero payment value     : "
    f"{(payments['payment_value'] == 0).sum():,}"
)

print(
    f"Installments < 1       : "
    f"{(payments['payment_installments'] < 1).sum():,}"
)

print("\nPayment types:")
print(payments["payment_type"].value_counts())

# ============================================================
# 12. REVIEW CHECK
# ============================================================

print("\n" + "=" * 80)
print("9. REVIEW CHECK")
print("=" * 80)

reviews = datasets["reviews"]

invalid_scores = (
    (~reviews["review_score"].isin([1, 2, 3, 4, 5]))
    | reviews["review_score"].isna()
).sum()

feedback_mismatch = (
    reviews["has_written_feedback"]
    != (
        reviews["review_comment_title"].notna()
        | reviews["review_comment_message"].notna()
    )
).sum()

print(f"Invalid / missing scores : {invalid_scores:,}")
print(f"Feedback flag mismatches : {feedback_mismatch:,}")

print("\nReview scores:")
print(
    reviews["review_score"]
    .value_counts()
    .sort_index()
)

print("\nWritten feedback:")
print(
    reviews["has_written_feedback"]
    .value_counts()
)

# ============================================================
# 13. PRODUCT CHECK
# ============================================================

print("\n" + "=" * 80)
print("10. PRODUCT CHECK")
print("=" * 80)

products = datasets["products"]

print(
    f"Negative weight        : "
    f"{(products['product_weight_g'] < 0).sum():,}"
)

print(
    f"Zero weight            : "
    f"{(products['product_weight_g'] == 0).sum():,}"
)

print(
    f"Negative length        : "
    f"{(products['product_length_cm'] < 0).sum():,}"
)

print(
    f"Negative height        : "
    f"{(products['product_height_cm'] < 0).sum():,}"
)

print(
    f"Negative width         : "
    f"{(products['product_width_cm'] < 0).sum():,}"
)

print(
    f"Missing English category: "
    f"{products['product_category_english'].isna().sum():,}"
)

# ============================================================
# 14. REFERENTIAL INTEGRITY
# ============================================================

print("\n" + "=" * 80)
print("11. REFERENTIAL INTEGRITY")
print("=" * 80)

orders = datasets["orders"]
items = datasets["order_items"]
payments = datasets["payments"]
reviews = datasets["reviews"]
products = datasets["products"]
sellers = datasets["sellers"]
customers = datasets["customers"]

relationships = {

    "orders → customers":
        (
            orders["customer_id"]
            .isin(customers["customer_id"])
        ).sum(),

    "order_items → orders":
        (
            items["order_id"]
            .isin(orders["order_id"])
        ).sum(),

    "order_items → products":
        (
            items["product_id"]
            .isin(products["product_id"])
        ).sum(),

    "order_items → sellers":
        (
            items["seller_id"]
            .isin(sellers["seller_id"])
        ).sum(),

    "payments → orders":
        (
            payments["order_id"]
            .isin(orders["order_id"])
        ).sum(),

    "reviews → orders":
        (
            reviews["order_id"]
            .isin(orders["order_id"])
        ).sum()
}

parent_sizes = {
    "orders → customers": len(orders),
    "order_items → orders": len(items),
    "order_items → products": len(items),
    "order_items → sellers": len(items),
    "payments → orders": len(payments),
    "reviews → orders": len(reviews)
}

for relationship, valid_count in relationships.items():

    total = parent_sizes[relationship]
    invalid = total - valid_count

    status = "✓" if invalid == 0 else "❌"

    print(
        f"{status} {relationship:<25} "
        f"Invalid references: {invalid:,}"
    )

# ============================================================
# 15. CATEGORY TRANSLATION CHECK
# ============================================================

print("\n" + "=" * 80)
print("12. CATEGORY TRANSLATION CHECK")
print("=" * 80)

translation = datasets["category_translation"]

duplicate_pt = translation[
    "product_category_name"
].duplicated().sum()

duplicate_en = translation[
    "product_category_name_english"
].duplicated().sum()

print(f"Duplicate Portuguese categories : {duplicate_pt:,}")
print(f"Duplicate English categories    : {duplicate_en:,}")

# ============================================================
# 16. FINAL DELIVERY DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("13. FINAL DELIVERY DISTRIBUTION")
print("=" * 80)

print(
    orders["delivery_performance"]
    .value_counts(dropna=False)
)

# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL VALIDATION SUMMARY")
print("=" * 80)

print("""
The cleaned dataset has been checked for:

✓ Expected files
✓ Required schema
✓ Duplicate records
✓ Primary/composite keys
✓ Null keys
✓ Order business rules
✓ Delivery metrics
✓ Delivery classification
✓ Item values
✓ Payment values
✓ Review scores
✓ Review feedback flag
✓ Product attributes
✓ Referential integrity
✓ Category translation
""")

print("=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)

FINAL CLEAN DATASET VALIDATION

Loading cleaned datasets...

✓ orders_clean.csv                         (99441, 16)
✓ order_items_clean.csv                    (112650, 7)
✓ payments_clean.csv                       (103886, 5)
✓ reviews_clean.csv                        (99224, 8)
✓ products_clean.csv                       (32951, 10)
✓ sellers_clean.csv                        (3095, 4)
✓ customers_clean.csv                      (99441, 5)
✓ category_translation_clean.csv           (71, 2)

1. FILE CHECK
✓ Exactly the expected 8 CSV files are present.

2. SCHEMA CHECK

ORDERS
✓ Required columns present

ORDER_ITEMS
✓ Required columns present

PAYMENTS
✓ Required columns present

REVIEWS
✓ Required columns present

PRODUCTS
✓ Required columns present

SELLERS
✓ Required columns present

CUSTOMERS
✓ Required columns present

CATEGORY_TRANSLATION
✓ Required columns present

3. BASIC DATASET CHECK

ORDERS
--------------------------------------------------
Rows        : 99,441
Columns     : 1